In [ ]:
!pip install torch torchvision torchaudio
!pip install tensorflow librosa opencv-python

In [ ]:
import torch
import tensorflow as tf
import librosa
import cv2
import numpy as np

In [ ]:
!pip install timm

In [ ]:
import torch
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"

model = timm.create_model("efficientnet_b0", pretrained=False)

# IMPORTANT: 2 outputs because checkpoint has 2 classes
model.classifier = torch.nn.Linear(model.classifier.in_features, 2)

checkpoint = torch.load("best_deepfake_model.pth", map_location=device)

model.load_state_dict(checkpoint)

model.to(device)
model.eval()

EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2

In [ ]:
def extract_video_features(x):

    features = model.forward_features(x)

    features = torch.nn.functional.adaptive_avg_pool2d(features, 1)

    features = torch.flatten(features, 1)

    return features

In [ ]:
checkpoint = torch.load("best_deepfake_model.pth")
print(type(checkpoint))
print(checkpoint.keys())

<class 'collections.OrderedDict'>
odict_keys(['conv_stem.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'blocks.0.0.conv_dw.weight', 'blocks.0.0.bn1.weight', 'blocks.0.0.bn1.bias', 'blocks.0.0.bn1.running_mean', 'blocks.0.0.bn1.running_var', 'blocks.0.0.bn1.num_batches_tracked', 'blocks.0.0.se.conv_reduce.weight', 'blocks.0.0.se.conv_reduce.bias', 'blocks.0.0.se.conv_expand.weight', 'blocks.0.0.se.conv_expand.bias', 'blocks.0.0.conv_pw.weight', 'blocks.0.0.bn2.weight', 'blocks.0.0.bn2.bias', 'blocks.0.0.bn2.running_mean', 'blocks.0.0.bn2.running_var', 'blocks.0.0.bn2.num_batches_tracked', 'blocks.1.0.conv_pw.weight', 'blocks.1.0.bn1.weight', 'blocks.1.0.bn1.bias', 'blocks.1.0.bn1.running_mean', 'blocks.1.0.bn1.running_var', 'blocks.1.0.bn1.num_batches_tracked', 'blocks.1.0.conv_dw.weight', 'blocks.1.0.bn2.weight', 'blocks.1.0.bn2.bias', 'blocks.1.0.bn2.running_mean', 'blocks.1.0.bn2.running_var', 'blocks.1.0.bn2.num_batches_tracked'

In [ ]:
import cv2
import numpy as np

img = cv2.imread("/content/fake_1.jpg")

img = cv2.resize(img,(224,224))

img = img / 255.0

In [ ]:
img = np.transpose(img,(2,0,1))   # HWC → CHW

img = torch.tensor(img).float()

image = img.unsqueeze(0).to(device)

In [ ]:
print(image.shape)

torch.Size([1, 3, 224, 224])


In [ ]:
with torch.no_grad():

    features = model.forward_features(image)

    features = torch.nn.functional.adaptive_avg_pool2d(features,1)

    features = torch.flatten(features,1)

print(features.shape)

torch.Size([1, 1280])


In [ ]:
print(features)

tensor([[-0.2049, -0.2595, -0.1337,  ..., -0.2626, -0.2346, -0.2488]],
       device='cuda:0')


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

audio_model = torch.load("deepfake_audio_model.pth", map_location=device)

audio_model.to(device)
audio_model.eval()

AttributeError: 'collections.OrderedDict' object has no attribute 'to'

In [ ]:
checkpoint = torch.load("deepfake_audio_model.pth")

print(type(checkpoint))
print(list(checkpoint.keys())[:10])

<class 'collections.OrderedDict'>
['features.0.weight', 'features.0.bias', 'features.2.weight', 'features.2.bias', 'features.5.weight', 'features.5.bias', 'features.7.weight', 'features.7.bias', 'features.10.weight', 'features.10.bias']


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load VGG16
audio_model = models.vgg16(pretrained=False)

# Same modification you used during training
audio_model.classifier[6] = nn.Linear(4096, 2)

# Load weights
state_dict = torch.load("deepfake_audio_model.pth", map_location=device)

audio_model.load_state_dict(state_dict)

audio_model = audio_model.to(device)

audio_model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
def extract_audio_features(x):

    x = audio_model.features(x)

    x = audio_model.avgpool(x)

    x = torch.flatten(x, 1)

    x = audio_model.classifier[:6](x)   # stop before last layer

    return x

In [ ]:
dummy = torch.randn(1,3,224,224).to(device)

audio_features = extract_audio_features(dummy)

print(audio_features.shape)

torch.Size([1, 4096])


In [ ]:
import tensorflow as tf

# path to the folder that contains saved_model.pb
# The previous path was incorrect as tf.saved_model.load expects the directory, not the file.
model_path = "/content/saved_model/saved_model_folder/"

image_model = tf.saved_model.load(model_path)

print(image_model.signatures)


_SignatureMap({'serve': <ConcreteFunction (*, input_layer: TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer')) -> Dict[['output_0', TensorSpec(shape=(None, 1), dtype=tf.float32, name='output_0')]] at 0x7910856DB5C0>, 'serving_default': <ConcreteFunction (*, input_layer: TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer')) -> Dict[['output_0', TensorSpec(shape=(None, 1), dtype=tf.float32, name='output_0')]] at 0x7910853A74D0>})


In [ ]:
import cv2
import numpy as np

img = cv2.imread("/content/fake_1.jpg")

img = cv2.resize(img,(224,224))

img = img / 255.0

img = np.expand_dims(img, axis=0)

print(img.shape)

(1, 224, 224, 3)


In [ ]:
infer = image_model.signatures["serving_default"]

# Cast the numpy array to float32 and pass it as a keyword argument 'input_layer'
output = infer(input_layer=tf.constant(img, dtype=tf.float32))

print(output)

{'output_0': <tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[0.00206237]], dtype=float32)>}


In [ ]:
image_features = list(output.values())[0]

print(image_features.shape)

(1, 1)


In [ ]:
import torch

# audio_features is already a tensor and likely on the correct device/dtype from previous operations
audio_features = audio_features.float().to(device)

# Corrected: Call 'extract_video_features' with the preprocessed 'image' tensor
video_features = extract_video_features(image).float().to(device)

image_features = torch.tensor(
    image_features.numpy()
).float().to(device)

In [ ]:
print("Audio:", audio_features.shape)
print("Video:", video_features.shape)
print("Image:", image_features.shape)

Audio: torch.Size([1, 4096])
Video: torch.Size([1, 1280])
Image: torch.Size([1, 1])


In [ ]:
combined_features = torch.cat(
    (audio_features, video_features, image_features),
    dim=1
)

print("Combined shape:", combined_features.shape)

Combined shape: torch.Size([1, 5377])


In [ ]:
import torch.nn as nn

fusion_model = nn.Sequential(

    nn.Linear(5377,1024),
    nn.ReLU(),

    nn.Linear(1024,256),
    nn.ReLU(),

    nn.Linear(256,1),
    nn.Sigmoid()

)

fusion_model = fusion_model.to(device)
fusion_model.eval()

Sequential(
  (0): Linear(in_features=5377, out_features=1024, bias=True)
  (1): ReLU()
  (2): Linear(in_features=1024, out_features=256, bias=True)
  (3): ReLU()
  (4): Linear(in_features=256, out_features=1, bias=True)
  (5): Sigmoid()
)

In [ ]:
with torch.no_grad():

    prediction = fusion_model(combined_features)

print("Fusion output:", prediction)

Fusion output: tensor([[0.4734]], device='cuda:0')


In [ ]:
print(audio_features.shape)
print(video_features.shape)
print(image_features.shape)
print(combined_features.shape)

torch.Size([1, 4096])
torch.Size([1, 1280])
torch.Size([1, 1])
torch.Size([1, 5377])


In [ ]:
torch.save(
    fusion_model.state_dict(),
    "fusion_model.pth"
)

https://drive.google.com/file/d/1tZ4K7VqDPPYI7BTSf52QpDfOPUi0ERpe/view?usp=sharing

In [ ]:
!gdown --id  "1tZ4K7VqDPPYI7BTSf52QpDfOPUi0ERpe"


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1tZ4K7VqDPPYI7BTSf52QpDfOPUi0ERpe
From (redirected): https://drive.google.com/uc?id=1tZ4K7VqDPPYI7BTSf52QpDfOPUi0ERpe&confirm=t&uuid=aeabdfd4-ed7e-4e72-a4f6-351270798d5b
To: /content/viddataset.zip
100% 2.92G/2.92G [00:55<00:00, 53.0MB/s]


In [ ]:
!unzip /content/viddataset.zip -d /content/dataset


Archive:  /content/viddataset.zip
   creating: /content/dataset/test/real/
  inflating: /content/dataset/test/real/00235.mp4  
  inflating: /content/dataset/test/real/00236.mp4  
  inflating: /content/dataset/test/real/00237.mp4  
  inflating: /content/dataset/test/real/00238.mp4  
  inflating: /content/dataset/test/real/00239.mp4  
  inflating: /content/dataset/test/real/00240.mp4  
  inflating: /content/dataset/test/real/00241.mp4  
  inflating: /content/dataset/test/real/00242.mp4  
  inflating: /content/dataset/test/real/00243.mp4  
  inflating: /content/dataset/test/real/00244.mp4  
  inflating: /content/dataset/test/real/00245.mp4  
  inflating: /content/dataset/test/real/00246.mp4  
  inflating: /content/dataset/test/real/00247.mp4  
  inflating: /content/dataset/test/real/00248.mp4  
  inflating: /content/dataset/test/real/00249.mp4  
  inflating: /content/dataset/test/real/00250.mp4  
  inflating: /content/dataset/test/real/00251.mp4  
  inflating: /content/dataset/test/real/0

In [ ]:
import os

In [ ]:
# 1. Install/Upgrade gdown
!pip install --upgrade gdown

import gdown

# 2. Define your folder link
url = 'https://drive.google.com/drive/folders/1gdC6WrttKyHLeObI4Nrp1rbVwT5UgDXk?usp=drive_link'

# 3. Use the --folder flag to download recursively
# Note: Google Drive folder downloads via link are limited to ~50 files.
# If your folder is larger, zip it first and download the zip link instead.
gdown.download_folder(url, quiet=True, use_cookies=False)

['/content/saved_model_folder/variables/variables.data-00000-of-00001',
 '/content/saved_model_folder/variables/variables.index',
 '/content/saved_model_folder/fingerprint.pb',
 '/content/saved_model_folder/saved_model.pb']

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load VGG16
audio_model = models.vgg16(pretrained=False)

# Same modification you used during training
audio_model.classifier[6] = nn.Linear(4096, 2)

# Load weights
state_dict = torch.load("deepfake_audio_model.pth", map_location=device)

audio_model.load_state_dict(state_dict)

audio_model = audio_model.to(device)

audio_model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
import torch
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"

video_model = timm.create_model("efficientnet_b0", pretrained=False)

# IMPORTANT: 2 outputs because checkpoint has 2 classes
video_model.classifier = torch.nn.Linear(model.classifier.in_features, 2)

checkpoint = torch.load("best_deepfake_model.pth", map_location=device)

video_model.load_state_dict(checkpoint)

video_model.to(device)
video_model.eval()

EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2

In [ ]:
import tensorflow as tf

# path to the folder that contains saved_model.pb
# The previous path was incorrect as tf.saved_model.load expects the directory, not the file.
model_path = "/content/saved_model_folder"

image_model = tf.saved_model.load(model_path)

print(image_model.signatures)


_SignatureMap({'serve': <ConcreteFunction (*, input_layer: TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer')) -> Dict[['output_0', TensorSpec(shape=(None, 1), dtype=tf.float32, name='output_0')]] at 0x7C5231948CB0>, 'serving_default': <ConcreteFunction (*, input_layer: TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer')) -> Dict[['output_0', TensorSpec(shape=(None, 1), dtype=tf.float32, name='output_0')]] at 0x7C521FD375F0>})


In [ ]:
import cv2
import numpy as np

img = cv2.imread("/content/fake_1.jpg")

img = cv2.resize(img,(224,224))

img = img / 255.0

img = np.expand_dims(img, axis=0)

print(img.shape)

error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


In [ ]:
image_features = list(output.values())[0]

print(image_features.shape)

NameError: name 'output' is not defined

In [ ]:
print("Audio:", get_audio_features.shape)
print("Video:", get_video_features.shape)
#print("Image:", image_features.shape)

AttributeError: 'function' object has no attribute 'shape'

In [ ]:
import librosa
import numpy as np
import cv2

def get_audio_features(audio_path):

    audio, sr = librosa.load(audio_path, sr=16000)

    spec = librosa.feature.melspectrogram(audio, sr)

    spec = cv2.resize(spec,(224,224))

    spec = np.stack((spec,)*3,axis=0)

    spec = torch.tensor(spec).unsqueeze(0).float()
    spec=spec.to(device)

    with torch.no_grad():

        x = audio_model.features(spec)

        x = torch.nn.functional.adaptive_avg_pool2d(x,1)

        features = torch.flatten(x,1)

    return features

In [ ]:
import cv2

def get_video_features(video_path):

    cap = cv2.VideoCapture(video_path)

    ret, frame = cap.read()

    cap.release()

    frame = cv2.resize(frame,(224,224))

    frame = frame/255.0

    frame = np.transpose(frame,(2,0,1))

    frame = torch.tensor(frame).unsqueeze(0).float()
    frame=frame.to(device)

    with torch.no_grad():

        x = video_model.forward_features(frame)

        x = torch.nn.functional.adaptive_avg_pool2d(x,1)

        features = torch.flatten(x,1)

    return features

In [ ]:
import os
import subprocess

def extract_audio_safe(video_path):

    audio_path = video_path.replace(".mp4",".wav")

    command = [
        "ffmpeg",
        "-i", video_path,
        "-vn",
        "-acodec","pcm_s16le",
        "-ar","16000",
        "-ac","1",
        audio_path,
        "-y"
    ]

    subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # check if audio was created
    if not os.path.exists(audio_path):
        return None

    return audio_path

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [ ]:
import os
import torch

features = []
labels = []

dataset_path = "dataset"

for label in ["train/real","train/fake"]:

    folder = os.path.join(dataset_path,label)

    for video in os.listdir(folder):

        if not video.endswith(".mp4"):
            continue

        video_path = os.path.join(folder,video)

        print("Processing:",video_path)

        # ---------- VIDEO FEATURES ----------
        video_feat = get_video_features(video_path)

        # ---------- AUDIO FEATURES ----------
        audio_path = extract_audio_safe(video_path)

        if audio_path is None:
            print("No audio found → using zero features")
            audio_feat = torch.zeros((1,512)).to(device)
        else:
            audio_feat = get_audio_features(audio_path)

        # ---------- IMAGE FEATURE ----------
        image_feat = torch.zeros((1,1)).to(device)

        # ---------- COMBINE ----------
        combined = torch.cat(
            (audio_feat,video_feat,image_feat),
            dim=1
        )

        features.append(combined.squeeze().cpu().numpy())

        labels.append(1 if label=="fake" else 0)

print("Total samples:",len(features))

Processing: dataset/train/real/id7_0000.mp4
No audio found → using zero features
Processing: dataset/train/real/00059.mp4
No audio found → using zero features
Processing: dataset/train/real/id45_0005.mp4
No audio found → using zero features
Processing: dataset/train/real/00061.mp4
No audio found → using zero features
Processing: dataset/train/real/id24_0002.mp4
No audio found → using zero features
Processing: dataset/train/real/id5_0004.mp4
No audio found → using zero features
Processing: dataset/train/real/00066.mp4
No audio found → using zero features
Processing: dataset/train/real/id17_0003.mp4
No audio found → using zero features
Processing: dataset/train/real/id48_0003.mp4
No audio found → using zero features
Processing: dataset/train/real/00064.mp4
No audio found → using zero features
Processing: dataset/train/real/id38_0003.mp4
No audio found → using zero features
Processing: dataset/train/real/id34_0001.mp4
No audio found → using zero features
Processing: dataset/train/real/id3

In [ ]:
# ===== Fusion Training Cell =====
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
from google.colab import files

# ---- Device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---- Convert features/labels to tensors ----
X = torch.tensor(np.array(features), dtype=torch.float32)
y = torch.tensor(np.array(labels), dtype=torch.float32).unsqueeze(1)

print("Dataset shape:", X.shape, y.shape)  # Expected: (N, 1793) (N, 1)

# ---- Train / Validation Split ----
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, y_train = X_train.to(device), y_train.to(device)
X_val, y_val = X_val.to(device), y_val.to(device)

# ---- Fusion Model ----
fusion_model = nn.Sequential(
    nn.Linear(1793, 512),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(128, 1),
    nn.Sigmoid()
).to(device)

# ---- Loss + Optimizer ----
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(fusion_model.parameters(), lr=5e-4)

# ---- Training Loop ----
epochs = 20
batch_size = 32

for epoch in range(epochs):
    fusion_model.train()

    perm = torch.randperm(X_train.size(0))
    epoch_loss = 0

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]

        xb = X_train[idx]
        yb = y_train[idx]

        optimizer.zero_grad()

        preds = fusion_model(xb)

        loss = criterion(preds, yb)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    # ---- Validation ----
    fusion_model.eval()
    with torch.no_grad():
        val_preds = fusion_model(X_val)
        val_loss = criterion(val_preds, y_val).item()

        val_acc = ((val_preds > 0.5) == y_val).float().mean().item()

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

# ---- Save Model ----
torch.save(fusion_model.state_dict(), "fusion_model.pth")
print("Fusion model saved!")

# ---- Download ----
files.download("fusion_model.pth")

Device: cuda
Dataset shape: torch.Size([1512, 1793]) torch.Size([1512, 1])
Epoch 1/20 | Train Loss: 2.5636 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 2/20 | Train Loss: 0.0002 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 3/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 4/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 5/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 6/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 7/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 8/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 9/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 10/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 11/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 12/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 13/20 | Train Loss: 0.0000 | Val Loss: 0.0000 | Val Acc: 1.0000
Epoch 14/20 | Train Loss

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Audio feature size:", audio_feat.shape)
print("Video feature size:", video_feat.shape)
print("Image feature size:", image_feat.shape)

combined = torch.cat((audio_feat, video_feat, image_feat), dim=1)

print("Combined feature size:", combined.shape)

Audio feature size: torch.Size([1, 512])
Video feature size: torch.Size([1, 1280])
Image feature size: torch.Size([1, 1])
Combined feature size: torch.Size([1, 1793])


In [ ]:
# take 10 samples
fusion_model.eval()

with torch.no_grad():

    sample_X = X[:10].to(device)
    sample_y = y[:10].to(device)

    preds = fusion_model(sample_X)

print("Predictions:", preds.squeeze().cpu().numpy())
print("Labels:", sample_y.squeeze().cpu().numpy())

Predictions: [1.8163709e-10 8.9582682e-12 8.7300256e-09 7.1804680e-12 7.8189268e-11
 1.9909806e-11 1.2258054e-11 7.5711069e-12 9.8710116e-11 5.7832447e-11]
Labels: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [ ]:
import torch

fusion_model.eval()

# random 10 samples
idx = torch.randperm(len(X))[:10]

sample_X = X[idx].to(device)
sample_y = y[idx].to(device)

with torch.no_grad():
    preds = fusion_model(sample_X)

print("Predictions:", preds.squeeze().cpu().numpy())
print("Labels:", sample_y.squeeze().cpu().numpy())

Predictions: [2.0681927e-11 1.9419313e-11 1.3415827e-11 2.1645926e-11 2.9359251e-11
 1.3139539e-11 1.4603046e-11 7.9770470e-12 3.2245262e-11 2.3003327e-11]
Labels: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [ ]:
print("REAL samples:", (y==0).sum().item())
print("FAKE samples:", (y==1).sum().item())

REAL samples: 1512
FAKE samples: 0


In [ ]:
features = []
labels = []

dataset_path = "dataset/train"

classes = ["real","fake"]

for label in classes:

    folder = os.path.join(dataset_path,label)

    print("Processing class:",label)

    for video in os.listdir(folder):

        if not video.endswith(".mp4"):
            continue

        video_path = os.path.join(folder,video)

        video_feat = get_video_features(video_path)

        audio_path = extract_audio_safe(video_path)

        if audio_path is None:
            audio_feat = torch.zeros((1,512)).to(device)
        else:
            audio_feat = get_audio_features(audio_path)

        image_feat = torch.zeros((1,1)).to(device)

        combined = torch.cat((audio_feat,video_feat,image_feat),dim=1)

        features.append(combined.squeeze().cpu().numpy())

        # label encoding
        if label == "real":
            labels.append(0)
        else:
            labels.append(1)

Processing class: real
Processing class: fake


In [ ]:
import torch

y = torch.tensor(labels)

print("REAL samples:", (y==0).sum().item())
print("FAKE samples:", (y==1).sum().item())

REAL samples: 756
FAKE samples: 756


In [ ]:
X = torch.tensor(np.array(features)).float()
y = torch.tensor(np.array(labels)).float().unsqueeze(1)

print("Feature shape:",X.shape)

Feature shape: torch.Size([1512, 1793])


In [ ]:
import torch.nn as nn
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train,X_val,y_train,y_val = train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

X_train = X_train.to(device)
X_val = X_val.to(device)

y_train = y_train.to(device)
y_val = y_val.to(device)

fusion_model = nn.Sequential(

    nn.Linear(1793,512),
    nn.ReLU(),

    nn.Linear(512,128),
    nn.ReLU(),

    nn.Linear(128,1),
    nn.Sigmoid()

).to(device)

optimizer = torch.optim.Adam(fusion_model.parameters(),lr=0.0005)

loss_fn = nn.BCELoss()

epochs = 20

for epoch in range(epochs):

    fusion_model.train()

    optimizer.zero_grad()

    preds = fusion_model(X_train)

    loss = loss_fn(preds,y_train)

    loss.backward()

    optimizer.step()

    # validation
    fusion_model.eval()

    with torch.no_grad():

        val_preds = fusion_model(X_val)

        val_loss = loss_fn(val_preds,y_val)

        val_acc = ((val_preds>0.5)==y_val).float().mean()

    print(
        f"Epoch {epoch+1} | Train Loss:{loss.item():.4f} | "
        f"Val Loss:{val_loss.item():.4f} | Val Acc:{val_acc.item():.4f}"
    )

Epoch 1 | Train Loss:0.6932 | Val Loss:0.6923 | Val Acc:0.5083
Epoch 2 | Train Loss:0.6916 | Val Loss:0.6927 | Val Acc:0.5017
Epoch 3 | Train Loss:0.6914 | Val Loss:0.6910 | Val Acc:0.5743
Epoch 4 | Train Loss:0.6894 | Val Loss:0.6908 | Val Acc:0.5182
Epoch 5 | Train Loss:0.6887 | Val Loss:0.6899 | Val Acc:0.5182
Epoch 6 | Train Loss:0.6871 | Val Loss:0.6897 | Val Acc:0.5149
Epoch 7 | Train Loss:0.6859 | Val Loss:0.6890 | Val Acc:0.5479
Epoch 8 | Train Loss:0.6843 | Val Loss:0.6882 | Val Acc:0.5578
Epoch 9 | Train Loss:0.6828 | Val Loss:0.6877 | Val Acc:0.5545
Epoch 10 | Train Loss:0.6812 | Val Loss:0.6875 | Val Acc:0.5743
Epoch 11 | Train Loss:0.6796 | Val Loss:0.6870 | Val Acc:0.5644
Epoch 12 | Train Loss:0.6778 | Val Loss:0.6867 | Val Acc:0.5611
Epoch 13 | Train Loss:0.6762 | Val Loss:0.6865 | Val Acc:0.5578
Epoch 14 | Train Loss:0.6744 | Val Loss:0.6870 | Val Acc:0.5677
Epoch 15 | Train Loss:0.6731 | Val Loss:0.6871 | Val Acc:0.5512
Epoch 16 | Train Loss:0.6716 | Val Loss:0.6875 | 

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------- Normalize features ----------
scaler = StandardScaler()
X_np = scaler.fit_transform(np.array(features))

X = torch.tensor(X_np).float()
y = torch.tensor(np.array(labels)).float().unsqueeze(1)

print("Feature shape:", X.shape)

# ---------- Train/Val Split ----------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train = X_train.to(device)
X_val = X_val.to(device)
y_train = y_train.to(device)
y_val = y_val.to(device)

# ---------- Improved Fusion Model ----------
fusion_model = nn.Sequential(

    nn.Linear(1793, 512),
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.4),

    nn.Linear(512, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(128, 1),
    nn.Sigmoid()

).to(device)

optimizer = torch.optim.Adam(fusion_model.parameters(), lr=0.0003)
criterion = nn.BCELoss()

epochs = 40
batch_size = 64

# ---------- Training Loop ----------
for epoch in range(epochs):

    fusion_model.train()

    perm = torch.randperm(X_train.size(0))
    epoch_loss = 0

    for i in range(0, X_train.size(0), batch_size):

        idx = perm[i:i+batch_size]

        xb = X_train[idx]
        yb = y_train[idx]

        optimizer.zero_grad()

        preds = fusion_model(xb)

        loss = criterion(preds, yb)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    # ---------- Validation ----------
    fusion_model.eval()

    with torch.no_grad():

        val_preds = fusion_model(X_val)

        val_loss = criterion(val_preds, y_val)

        val_acc = ((val_preds > 0.5) == y_val).float().mean()

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss:{epoch_loss:.4f} | "
        f"Val Loss:{val_loss:.4f} | "
        f"Val Acc:{val_acc:.4f}"
    )

# ---------- Save Model ----------
torch.save(fusion_model.state_dict(), "fusion_model.pth")

print("Fusion model saved!")

Device: cuda
Feature shape: torch.Size([1512, 1793])
Epoch 1/40 | Train Loss:12.3711 | Val Loss:0.6069 | Val Acc:0.7162
Epoch 2/40 | Train Loss:10.4418 | Val Loss:0.5380 | Val Acc:0.7690
Epoch 3/40 | Train Loss:9.1698 | Val Loss:0.4819 | Val Acc:0.8086
Epoch 4/40 | Train Loss:8.2095 | Val Loss:0.4362 | Val Acc:0.8515
Epoch 5/40 | Train Loss:7.4987 | Val Loss:0.3975 | Val Acc:0.8680
Epoch 6/40 | Train Loss:7.0086 | Val Loss:0.3715 | Val Acc:0.8614
Epoch 7/40 | Train Loss:7.0670 | Val Loss:0.3556 | Val Acc:0.8977
Epoch 8/40 | Train Loss:6.3041 | Val Loss:0.3481 | Val Acc:0.9076
Epoch 9/40 | Train Loss:5.7520 | Val Loss:0.3238 | Val Acc:0.8977
Epoch 10/40 | Train Loss:5.5433 | Val Loss:0.3284 | Val Acc:0.9076
Epoch 11/40 | Train Loss:5.5274 | Val Loss:0.3215 | Val Acc:0.8944
Epoch 12/40 | Train Loss:5.1562 | Val Loss:0.3302 | Val Acc:0.8911
Epoch 13/40 | Train Loss:5.0049 | Val Loss:0.2950 | Val Acc:0.9010
Epoch 14/40 | Train Loss:5.1807 | Val Loss:0.2821 | Val Acc:0.9043
Epoch 15/40 | Tr

In [ ]:
from google.colab import files
files.download("fusion_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>